In [ ]:
import cupy as cp
import torch
import triton
import triton.language as tl

In [ ]:
def naive_softmax(x):

    row_max = cp.max(x, axis=-1, keepdims=True)
    x_stable = x - roww_max
    x_exp = cp.exp(x_stable)
    row_sum = cp.sum(x_exp, axis=-1, keepdims=True)
    out = x_exp / row_sum
    return out



In [ ]:
# (n_rows, n_cols)
@triton.jit
def softmax_kernel_forward(
    output_ptr,
    input_ptr,
    input_row_stride,
    output_row_stride,
    DTYPE_FLAG: tl.constexpr,
    n_cols: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
):
    row_idx = tl.program_id(0)

    #### extra ####
    pointer_dtype = tl.float32 if DTYPE_FLAG == 0 else tl.float16
    input_ptr = tl.cast(input_ptr, tl.pointer_type(pointer_dtype))
    output_ptr = tl.cast(output_ptr, tl.pointer_type(pointer_dtype))
    #######

    row_start_ptr = input_ptr + row_idx * input_row_stride
    col_offsets = tl.arange(0, BLOCK_SIZE)
    input_ptrs = row_start_ptr + col_offsets

    # BLOCK_SIZE must be a power of 2
    mask = col_offsets < n_cols

    row = tl.load(input_ptrs, mask=mask, other=-float("inf"))

    row_minus_max = row - tl.max(row, axis=0)
    numerator = tl.exp(row_minus_max)
    denominator = tl.sum(numerator, axis=0)
    softmax_output = (numerator / denominator).to(output_ptr.dtype.element_ty)

    output_row_start_ptr = output_ptr + row_idx * output_row_stride
    output_ptrs = output_row_start_ptr + col_offsets
    tl.store(output_ptrs, softmax_output, mask=mask)



In [ ]:
@triton.jit
def softmax_kernel_backward(
        output_ptr, output_row_stride,
        s_ptr, s_row_stride,
        grad_ptr, grad_row_stride,
        DTYPE_FLAG: tl.constexpr,
        n_cols: tl.constexpr,
        BLOCK_SIZE: tl.constexpr,
):
    row_idx = tl.program_id(0)

    ### extra for cupy ###
    pointer_dtype = tl.float32 if DTYPE_FLAG == 0 else tl.float16
    s_ptr = tl.cast(s_ptr, tl.pointer_type(pointer_dtype))
    grad_ptr = tl.cast(grad_ptr, tl.pointer_type(pointer_dtype))
    output_ptr = tl.cast(output_ptr, tl.pointer_type(pointer_dtype))
    ######################

    s_row_start_ptr = s_ptr + row_idx * s_row_stride
    grad_row_start_ptr = grad_ptr + row_idx * grad_row_stride

    col_offsets = tl.arange(0, BLOCK_SIZE)
    s_ptrs = s_row_start_ptr + col_offsets
    grad_ptrs = grad_row_start_ptr + col_offsets

    mask = col_offsets < n_cols

    s_row = tl.load(s_ptrs, mask=mask, other=0.0)
    grad_row = tl.load(grad_ptrs, mask=mask, other=0.0)

    # s * (grad - sum(grad*s)) -> s*grad - s*sum(s*grad)
    grad_x_s = grad_row * s_row
    softmax_grad_out = grad_x_s - s_row * tl.sum(grad_x_s, axis=0)

    output_row_start_ptr = output_ptr + row_idx * output_row_stride
    output_ptrs = output_row_start_ptr + col_offsets
    tl.store(output_ptrs, softmax_grad_out, mask=mask)


In [ ]:
def fused_softmax_forward(x, use_dlpack=True):

    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)

    if not DLPACK_DISABLE or use_dlpack:

        # cupy -> torch -> cupy
        x = torch.utils.dlpack.from_dlpack(x)
        y = torch.empty_like(x)

        dtype_flag = 0 if x.dtype == torch.float32 else 1   # 0=float32, 1=float16
        row_stride = x.stride(0)

        grid = (n_rows,)
        softmax_kernel_forward[grid](
            y,
            x,
            row_stride,
            row_stride,
            dtype_flag,
            n_cols,
            BLOCK_SIZE
        )

        return cp.from_dlpack(y)


    else:
        y = cp.empty_like(x)

        # compute strides in elements
        row_stride = x.strides[0] // x.itemsize

        # map dtype to Triton flag
        dtype_flag = 0 if x.dtype == cp.float32 else 1      # 0 = float32, 1 = float16

        grid = (n_rows, )
        softmax_kernel_forward[grid](
            y.data.ptr,
            x.data.ptr,
            row_stride,
            row_stride,
            dtype_flag,
            n_cols,
            BLOCK_SIZE
        )

        return y

In [ ]:
def fused_softmax_backward(grad, s, use_dlpack=True):

    n_rows, n_cols = grad.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)

    if not DLPACK_DISABLE and use_dlpack:

        grad = torch.utils.dlpack.from_dlpack(grad)
        s = torch.utils.dlpack.from_dlpack(s)

        grad_input = torch.empty_like(grad)
        row_stride = grad.stride(0)
        dtype_flag = 0 if grad.dtype == torch.float32 else 1

        grid = (n_rows,)
        softmax_kernel_backward[grid](
            grad_input,
            row_stride,
            s,
            row_stride,
            grad,
            row_stride,
            dtype_flag,
            n_cols,
            BLOCK_SIZE
        )

        return cp.from_dlpack(grad_input)

    else:
        grad_input = cp.empty_like(grad)

        row_stride = grad.strides[0] // grad.itemsize

        dtype_flag = 0 if grad.dtype == cp.float32 else 1

        grid = (n_rows, )
        softmax_kernel_backward[grid](
            grad_input.data.ptr,
            row_stride,
            s.data.ptr,
            row_stride,
            grad.data.ptr,
            row_stride,
            dtype_flag,
            n_cols,
            BLOCK_SIZE,
        )
        return grad_input

